# Lightweight RAG: Course Syllabus Chatbot

This notebook implements a small retrieval pipeline using a course syllabus as plain text.

**Pipeline:** Text Chunking → Sentence Embeddings → Cosine Similarity → Retrieve Relevant Chunks → Extractive Summary

This is a lightweight RAG-style prototype. The final summary is extractive (selected from retrieved text), not generated by an LLM.

## 1. Install and import libraries

If the libraries are not installed in your environment, run the installation cell first.


In [1]:
%pip install -q sentence-transformers scikit-learn numpy

In [2]:
import re
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

## 2. Provide the course syllabus

For this demonstration, the syllabus is stored as a Python string. You can replace it with the text extracted from a `.txt` file or another document.


In [3]:
syllabus = """
INTRODUCTION TO MACHINE LEARNING — COURSE SYLLABUS

Course Information
Course code: ML101. Credits: 3. Duration: 15 weeks.
Prerequisites: Basic Python programming, introductory statistics, and fundamental linear algebra.

Course Description
This course introduces machine learning concepts, algorithms, and applications. Students will prepare datasets, train models, evaluate performance, and use Python to solve practical problems.

Course Objectives
By the end of the course, students should be able to explain major types of machine learning, clean and transform datasets, implement basic algorithms, evaluate models, recognize overfitting, and complete a small machine learning project.

Textbooks and Resources
The main textbook is Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow by Aurélien Géron. An additional resource is An Introduction to Statistical Learning by Gareth James, Daniela Witten, Trevor Hastie, Robert Tibshirani, and Jonathan Taylor. Lecture notes, programming notebooks, and instructor-provided datasets are also used.

Course Modules
Module 1: Introduction to Machine Learning.
Module 2: Python and Data Preparation.
Module 3: Exploratory Data Analysis.
Module 4: Feature Engineering and Feature Selection.
Module 5: Supervised Learning — Regression.
Module 6: Supervised Learning — Classification.
Module 7: Unsupervised Learning and Clustering.
Module 8: Model Evaluation and Cross-Validation.
Module 9: Neural Networks and Deep Learning.
Module 10: Machine Learning Project and Applications.

Assessment and Grading
Programming labs account for 15% of the final grade. Quizzes account for 10%. The midterm test accounts for 15%. The practical project accounts for 25%. The final examination accounts for 35%. The total is 100%.

Examination and Project Requirements
The midterm test covers Modules 1–5 and assesses conceptual understanding and basic problem-solving. Students complete an individual or pair-based practical project using a dataset. The project includes data preparation, model training, evaluation, and a written report. The final examination covers all ten modules, with emphasis on algorithm selection, model evaluation, data preparation, and interpreting results.

Learning Expectations
Students should attend lectures, complete programming labs, participate in quizzes, and submit assessments before deadlines. Basic familiarity with Python is expected from the beginning of the course.
"""

print(syllabus[:500], "...")



INTRODUCTION TO MACHINE LEARNING — COURSE SYLLABUS

Course Information
Course code: ML101. Credits: 3. Duration: 15 weeks.
Prerequisites: Basic Python programming, introductory statistics, and fundamental linear algebra.

Course Description
This course introduces machine learning concepts, algorithms, and applications. Students will prepare datasets, train models, evaluate performance, and use Python to solve practical problems.

Course Objectives
By the end of the course, students should be ab ...


## 3. Text chunking

Chunking divides a long document into smaller pieces so the retriever can find specific information without treating the entire syllabus as one unit. This example uses simple paragraph-based chunking, preserving section boundaries where possible.

For a real project, recursive chunking and a small overlap can help preserve context when sections are long. Here we keep the first implementation simple.


In [4]:
def chunk_text(text: str) -> list[str]:
    """Split text into non-empty paragraph chunks."""
    chunks = [part.strip() for part in re.split(r"\n\s*\n", text) if part.strip()]
    return chunks

chunks = chunk_text(syllabus)
print(f"Number of chunks: {len(chunks)}\n")
for i, chunk in enumerate(chunks, start=1):
    print(f"Chunk {i}: {chunk[:160]}{'...' if len(chunk) > 160 else ''}\n")


Number of chunks: 9

Chunk 1: INTRODUCTION TO MACHINE LEARNING — COURSE SYLLABUS

Chunk 2: Course Information
Course code: ML101. Credits: 3. Duration: 15 weeks.
Prerequisites: Basic Python programming, introductory statistics, and fundamental linear ...

Chunk 3: Course Description
This course introduces machine learning concepts, algorithms, and applications. Students will prepare datasets, train models, evaluate perfor...

Chunk 4: Course Objectives
By the end of the course, students should be able to explain major types of machine learning, clean and transform datasets, implement basic al...

Chunk 5: Textbooks and Resources
The main textbook is Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow by Aurélien Géron. An additional resource is An ...

Chunk 6: Course Modules
Module 1: Introduction to Machine Learning.
Module 2: Python and Data Preparation.
Module 3: Exploratory Data Analysis.
Module 4: Feature Enginee...

Chunk 7: Assessment and Grading
Programming labs

## 4. Build the lightweight RAG class

The class below handles three jobs:

- **Indexing:** create an embedding for every syllabus chunk.
- **Retrieval:** embed a query and rank chunks by cosine similarity.
- **Summary:** return the highest-ranked chunks as an extractive summary.

We use `sentence-transformers/all-MiniLM-L6-v2`, a compact sentence-embedding model. The embedding model does not write the summary; it helps identify semantically relevant text.


In [5]:
class SimpleRAG:
    def __init__(self, text: str, model_name: str = "sentence-transformers/all-MiniLM-L6-v2"):
        self.chunks = chunk_text(text)
        if not self.chunks:
            raise ValueError("The syllabus contains no text to index.")

        self.model = SentenceTransformer(model_name)
        # Convert every chunk into an embedding once during initialization.
        self.chunk_embeddings = self.model.encode(
            self.chunks,
            convert_to_numpy=True,
            normalize_embeddings=True
        )

    def retrieve(self, query: str, top_k: int = 3):
        """Return the top-k chunks most similar to the query."""
        if not query.strip():
            raise ValueError("Please enter a non-empty query.")
        query_embedding = self.model.encode(
            [query],
            convert_to_numpy=True,
            normalize_embeddings=True
        )
        scores = cosine_similarity(query_embedding, self.chunk_embeddings)[0]
        ranked_indices = np.argsort(scores)[::-1][:max(1, min(top_k, len(self.chunks)))]
        return [(self.chunks[i], float(scores[i])) for i in ranked_indices]

    def summarize(self, query: str, top_k: int = 3) -> str:
        """Create a simple extractive summary from retrieved chunks."""
        results = self.retrieve(query, top_k=top_k)
        return "\n\n".join(chunk for chunk, score in results)

rag = SimpleRAG(syllabus)
print(f"Indexed {len(rag.chunks)} chunks.")
print(f"Embedding dimensions: {rag.chunk_embeddings.shape[1]}")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Indexed 9 chunks.
Embedding dimensions: 384


## 5. Retrieve the most relevant chunks

Try a question about the syllabus. The similarity score indicates how closely each chunk matches the query in embedding space; it is not a probability that the chunk is correct.


In [6]:
query = "How is the final grade calculated, and how much is the project worth?"

results = rag.retrieve(query, top_k=3)
for rank, (chunk, score) in enumerate(results, start=1):
    print(f"Result {rank} | cosine similarity: {score:.3f}")
    print(chunk)
    print("-" * 80)


Result 1 | cosine similarity: 0.559
Assessment and Grading
Programming labs account for 15% of the final grade. Quizzes account for 10%. The midterm test accounts for 15%. The practical project accounts for 25%. The final examination accounts for 35%. The total is 100%.
--------------------------------------------------------------------------------
Result 2 | cosine similarity: 0.419
Examination and Project Requirements
The midterm test covers Modules 1–5 and assesses conceptual understanding and basic problem-solving. Students complete an individual or pair-based practical project using a dataset. The project includes data preparation, model training, evaluation, and a written report. The final examination covers all ten modules, with emphasis on algorithm selection, model evaluation, data preparation, and interpreting results.
--------------------------------------------------------------------------------
Result 3 | cosine similarity: 0.284
Course Information
Course code: ML101. Cr

## 6. Retrieve a summary

Our lightweight version calls the concatenation of the most relevant chunks a summary. It is extractive: the returned wording comes directly from the syllabus. It may include more detail than a polished LLM-written summary, but it is easy to inspect and trace back to the source.


In [7]:
query = "Summarize the course modules, learning objectives, and assessment methods."
answer = rag.summarize(query, top_k=4)
print("Retrieved summary:\n")
print(answer)

Retrieved summary:

Course Modules
Module 1: Introduction to Machine Learning.
Module 2: Python and Data Preparation.
Module 3: Exploratory Data Analysis.
Module 4: Feature Engineering and Feature Selection.
Module 5: Supervised Learning — Regression.
Module 6: Supervised Learning — Classification.
Module 7: Unsupervised Learning and Clustering.
Module 8: Model Evaluation and Cross-Validation.
Module 9: Neural Networks and Deep Learning.
Module 10: Machine Learning Project and Applications.

Examination and Project Requirements
The midterm test covers Modules 1–5 and assesses conceptual understanding and basic problem-solving. Students complete an individual or pair-based practical project using a dataset. The project includes data preparation, model training, evaluation, and a written report. The final examination covers all ten modules, with emphasis on algorithm selection, model evaluation, data preparation, and interpreting results.

Course Description
This course introduces machin

## 7. Ask your own question

Change the query and `top_k` value to explore different parts of the syllabus.


In [8]:
query = "What are the prerequisites and required textbooks?"
print(rag.summarize(query, top_k=2))

Course Information
Course code: ML101. Credits: 3. Duration: 15 weeks.
Prerequisites: Basic Python programming, introductory statistics, and fundamental linear algebra.

Textbooks and Resources
The main textbook is Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow by Aurélien Géron. An additional resource is An Introduction to Statistical Learning by Gareth James, Daniela Witten, Trevor Hastie, Robert Tibshirani, and Jonathan Taylor. Lecture notes, programming notebooks, and instructor-provided datasets are also used.


## What this notebook demonstrates

- Paragraph chunking creates smaller retrievable units.
- Sentence Transformers embeds the chunks and user query in the same vector space.
- Scikit-learn cosine similarity ranks chunks by semantic similarity.
- The `SimpleRAG` class retrieves relevant text and returns an extractive summary.

**Limitation:** This is a minimal retrieval-and-context prototype, not a full generative RAG system. To generate a fluent, newly written summary, pass the retrieved chunks and the user's query to an LLM, and instruct it to answer only from the supplied context. For production use, also consider chunk overlap, source metadata, access control, evaluation, and handling questions for which no relevant passage is found.
